In [2]:
from PIL import Image, ImageDraw, ImageFont
from pdf2image import convert_from_path
import os

# ----------------- Einstellungen -----------------
root = "/home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing"
base_teaser = os.path.join(root, "figures", "teaser")
base_co    = os.path.join(root, "figures", "co_d0")      # <-- color-opponency folder from your LaTeX
base_skala = os.path.join(root, "figures", "teaser", "skala")
out_png = "color-opponency_figure.png"
out_pdf = "color-opponency_figure.pdf"
poppler_path = None  # optional z.B. "/usr/bin"
crop_factor = 0.03

# Layout
orig_w = 540       # Breite Original Image
slice_w = 180      # Breite jeder Slice
slice_h = 360
label_w = 60
col_gap = 8
row_gap = 12
bg = (255,255,255)

# helper path builders
def p_teaser(*parts):
    return os.path.join(base_teaser, *parts)
def p_co(*parts):
    return os.path.join(base_co, *parts)
def p_skala(*parts):
    return os.path.join(base_skala, *parts)

# ----------------- file_map angepasst an dein LaTeX (co_d0 Pfade) -----------------
file_map = {
    ("Cityscapes", "Orig Image"): [ p_teaser("image_beforeRGB.pdf") ],
    ("Cityscapes", "Color-opponency"): [
        p_co("image_beforeRGB-1co_d0_after-r-i.pdf"),
        p_co("image_beforeRGB-1co_d0_after-g.pdf"),
        p_co("image_beforeRGB-1_-3co_d0_after-b.pdf"),
    ],

    ("Dark Zurich", "Orig Image"): [ p_teaser("image_before_night_dark_zurichRGB.pdf") ],
    ("Dark Zurich", "Color-opponency"): [
        p_co("image_before_dark_zurich_RGB-1co_d0_after-r-i.pdf"),
        p_co("image_before_dark_zurich_RGB-1co_d0_after-g.pdf"),
        p_co("image_before_dark_zurich_RGB-1_-3co_d0_after-b.pdf"),
    ],

    ("ACDC Night", "Orig Image"): [ p_teaser("image_before_acdc_nightRGB.pdf") ],
    ("ACDC Night", "Color-opponency"): [
        p_co("image_before_acdc_night_RGB-1co_d0_after-r-i.pdf"),
        p_co("image_before_acdc_night_RGB-1co_d0_after-g.pdf"),
        p_co("image_before_acdc_night_RGB-1_-3co_d0_after-b.pdf"),
    ],

    ("ACDC Snow", "Orig Image"): [ p_teaser("image_before_acdc_snowRGB.pdf") ],
    ("ACDC Snow", "Color-opponency"): [
        p_co("image_before_acdc_snow_RGB-1co_d0_after-r-i.pdf"),
        p_co("image_before_acdc_snow_RGB-1co_d0_after-g.pdf"),
        p_co("image_before_acdc_snow_RGB-1_-3co_d0_after-b.pdf"),
    ],

    ("ACDC Fog", "Orig Image"): [ p_teaser("image_before_acdc_fogRGB.pdf") ],
    ("ACDC Fog", "Color-opponency"): [
        p_co("image_before_acdc_fog_RGB-1co_d0_after-r-i.pdf"),
        p_co("image_before_acdc_fog_RGB-1co_d0_after-g.pdf"),
        p_co("image_before_acdc_fog_RGB-1_-3co_d0_after-b.pdf"),
    ],

    ("ACDC Rain", "Orig Image"): [ p_teaser("image_before_acdc_rainRGB.pdf") ],
    ("ACDC Rain", "Color-opponency"): [
        p_co("image_before_acdc_rain_RGB-1co_d0_after-r-i.pdf"),
        p_co("image_before_acdc_rain_RGB-1co_d0_after-g.pdf"),
        p_co("image_before_acdc_rain_RGB-1co_d0_after-b.pdf"),
    ],

    # Skala row (wie in LaTeX last line)
    ("Skala", "Orig Image"): [ p_teaser("skala", "rgb-wide.pdf") ],
    ("Skala", "Color-opponency"): [
        p_teaser("skala", "bla-w-num.pdf"),
        p_teaser("skala", "g-w-r-num.pdf"),
        p_teaser("skala", "y-w-b-num.pdf"),
    ],
}

# Fonts
try:
    font_path = "/home/lstracke/Data/times.ttf"
    font_label  = ImageFont.truetype(font_path, 40)
    font_small  = ImageFont.truetype(font_path, 40)
except Exception:
    font_label  = ImageFont.load_default()
    font_small  = ImageFont.load_default()

# ----------------- Cropping & loading helpers -----------------
def crop_image_mask(im, crop_factor=crop_factor):
    w,h = im.size
    l = int(w*crop_factor); t = int(h*crop_factor)
    r = int(w*(1-crop_factor)); b = int(h*(1-crop_factor))
    if r-l>5 and b-t>5:
        return im.crop((l,t,r,b))
    return im

def crop_image_slice(im, channel, crop_factor=crop_factor):
    w,h = im.size
    l,t,r,b = 0,0,w,h
    if channel=='R':
        l=int(w*crop_factor); t=int(h*crop_factor); b=int(h*(1-crop_factor))
    elif channel=='G':
        t=int(h*crop_factor); b=int(h*(1-crop_factor))
    elif channel=='B':
        t=int(h*crop_factor); b=int(h*(1-crop_factor)); r=int(w*(1-crop_factor))
    if r-l>5 and b-t>5:
        return im.crop((l,t,r,b))
    return im

def load_any(path, slice_channel=None, is_orig=False, dpi=150):
    """Lädt PDF/IMG, cropt je nach Art, und loggt was geladen wurde."""
    if not path or not os.path.exists(path):
        print(f"[MISSING] {path}")
        return None
    try:
        if path.lower().endswith(".pdf"):
            pages = convert_from_path(path, dpi=dpi, poppler_path=poppler_path) if poppler_path else convert_from_path(path, dpi=dpi)
            im = pages[0].convert("RGB")
        else:
            im = Image.open(path).convert("RGB")

        # Debug: show which file loaded
        print(f"[LOADED] {path}  size={im.size}")

        if "mask" in path.lower():
            im = crop_image_mask(im)
            print(f"  -> cropped mask -> {im.size}")
        elif slice_channel is not None:
            im = crop_image_slice(im, slice_channel)
            print(f"  -> cropped slice {slice_channel} -> {im.size}")
        elif is_orig:
            im = crop_image_mask(im)
            print(f"  -> cropped orig -> {im.size}")
        return im
    except Exception as e:
        print(f"[WARN] {path} konnte nicht geladen werden: {e}")
        return None

def make_placeholder(w,h,text="",font=font_small):
    im = Image.new("RGB",(w,h),(240,240,240))
    d = ImageDraw.Draw(im)
    bbox = d.textbbox((0,0), text, font=font)
    tw = bbox[2]-bbox[0]; th = bbox[3]-bbox[1]
    d.text(((w-tw)//2,(h-th)//2), text, font=font, fill=(0,0,0))
    return im

def concat_horiz(images, bg=bg):
    total_w = sum(im.width for im in images)
    h = max(im.height for im in images)
    out = Image.new("RGB",(total_w,h),bg)
    x=0
    for im in images:
        out.paste(im,(x,(h-im.height)//2))
        x+=im.width
    return out

# ----------------- Rows (LaTeX order) -----------------
rows = ["Cityscapes","Dark Zurich","ACDC Night","ACDC Fog","ACDC Rain","ACDC Snow","Skala"]
channels = ['R','G','B']

# ----------------- Build cells -----------------
cells_orig = {}
cells_slices = {}
cells_labels = {}

for row in rows:
    # Orig
    f_orig = file_map.get((row,"Orig Image"), [None])[0]
    im_orig = load_any(f_orig, is_orig=True) if f_orig else None
    if im_orig is None:
        im_orig = make_placeholder(orig_w, slice_h, text=f"(no orig: {row})")
    im_orig = im_orig.resize((orig_w, slice_h), Image.LANCZOS)
    cells_orig[row] = im_orig

    # Color-opponency: expect exact 3 files; if missing fill placeholders
    slice_files = file_map.get((row,"Color-opponency"), [])
    slice_imgs=[]
    for i in range(3):
        if i < len(slice_files):
            f_s = slice_files[i]
            im_s = load_any(f_s, slice_channel=channels[i])
            if im_s is None:
                im_s = make_placeholder(slice_w, slice_h, text=os.path.basename(f_s))
            else:
                im_s = im_s.resize((slice_w, slice_h), Image.LANCZOS)
        else:
            im_s = make_placeholder(slice_w, slice_h, text="(missing slice)")
        slice_imgs.append(im_s)
    cells_slices[row] = concat_horiz(slice_imgs)

    # Label (vertical)
    label_img = Image.new("RGBA",(slice_h,label_w),(0,0,0,0))
    d = ImageDraw.Draw(label_img)
    bbox = d.textbbox((0,0), row, font=font_label)
    tw = bbox[2]-bbox[0]; th = bbox[3]-bbox[1]
    d.text(((slice_h-tw)//2,(label_w-th)//2), row, font=font_label, fill=(0,0,0))
    label_img = label_img.rotate(90, expand=True)
    cells_labels[row] = label_img

# ----------------- Canvas erstellen -----------------
W = label_w + orig_w + slice_w*3 + col_gap
scale_h = 25  # Höhe der Skala
num_normal_rows = len(rows) - 1  # alle außer Skala
H = num_normal_rows * slice_h + (num_normal_rows) * row_gap + scale_h + row_gap
canvas = Image.new("RGB", (W, H), bg)

y = 0
for idx, row in enumerate(rows):
    # Label
    canvas.paste(cells_labels[row], (0, y), cells_labels[row])
    x = label_w
    # Original
    im_orig = cells_orig[row]
    if row == "Skala":
        # Original-Skala proportional auf scale_h
        w, h = im_orig.size
        new_w = orig_w  # volle Original-Breite
        im_orig = im_orig.resize((new_w, scale_h), Image.LANCZOS)
        canvas.paste(im_orig, (x, y))
        x += orig_w + col_gap

        # Color-opponency-Skala
        im_slice = cells_slices[row]
        im_slice = im_slice.resize((slice_w*3, scale_h), Image.LANCZOS)
        canvas.paste(im_slice, (x, y))

        y += scale_h + row_gap
    else:
        # normale Rows
        canvas.paste(im_orig, (x, y))
        x += orig_w + col_gap
        canvas.paste(cells_slices[row], (x, y))
        y += slice_h + row_gap



# ----------------- Save -----------------
canvas.save(out_png)
canvas.save(out_pdf, "PDF", resolution=300)
print("Fertig:", out_png, out_pdf)

[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/teaser/image_beforeRGB.pdf  size=(4267, 2134)
  -> cropped orig -> (4010, 2005)
[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/co_d0/image_beforeRGB-1co_d0_after-r-i.pdf  size=(1096, 1755)
  -> cropped slice R -> (1064, 1650)
[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/co_d0/image_beforeRGB-1co_d0_after-g.pdf  size=(1096, 1755)
  -> cropped slice G -> (1096, 1650)
[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/co_d0/image_beforeRGB-1_-3co_d0_after-b.pdf  size=(1169, 1755)
  -> cropped slice B -> (1133, 1650)
[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/teaser/image_before_night_dark_zurichRGB.pdf  size=(4001, 2250)
  -> cropped orig -> (3760, 2115)
[LOADED] /home/lstracke/Downloads/ECCV_2026_Paper_Preprocessing/figures/co_d0/image_before_dark_zurich_RGB-1co_d0_after-r-i.pdf  size=(1040, 1755)
  -> cropped slice 